# Architecture ablation

Reproduces Table II and Figure 3: the cumulative ablation on NKI from the default CNN of Zunair et al. to the un-CNN. Each step adds one change to the previous configuration. All architectural and preprocessing choices were made on NKI only, under this setup: rigidly registered TurboPrep volumes and a random forest head. These numbers are therefore not comparable with the main benchmark.

Every variant is built with seed 0, frozen and run in eval mode, so dropout is inactive and BatchNorm applies its initial running statistics. The ablation runtime of a step is the wall clock time to extract the features of all subjects and evaluate both tasks over the five seeds, divided by the number of subjects. `slurm/architecture_ablation.sh` runs the notebook.

Outputs in `outputs/architecture_ablation/`: `ablation_results.csv`, every score per step, task and seed, and `steps.csv`, one row per step.

In [ ]:
import os

N_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))

In [ ]:
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[var] = str(N_CPUS)

import gc
import time

import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

import cohorts
from config import OUTPUT_DIR
from uncnn import DepthwiseSepConv3d, UnCNN, clip_minmax, load_and_resize, norm, rank_filter, scale_robust, sobel_mag

torch.set_num_threads(N_CPUS)
torch.set_num_interop_threads(N_CPUS)

OUT = OUTPUT_DIR / "architecture_ablation"
OUT.mkdir(parents=True, exist_ok=True)
print(f"{N_CPUS} threads")

## Subjects

NKI subjects with a rigid TurboPrep volume, a CAT12 gray matter map and complete FreeSurfer features.

In [ ]:
subjects, files, sex, age = cohorts.load_nki_rigid()

## Inputs

Every volume is first resized to 91 × 109 × 91. Steps 1 to 3 use the z scored volume, as in the foundation model benchmark where the untrained CNN was first observed. Step 4 clips intensities at the 1st and 99th percentiles and min max scales them. Step 6 adds two channels: an unsharp masked image, $x + (x - G_1 * x)$ clipped to [0, 1], and the Sobel gradient magnitude. Step 8 replaces unsharp masking with a 3 × 3 × 3 median filter, step 9 clips at the 2nd and 98th percentiles, and step 11 replaces clipping and min max scaling with the robust scaling of the un-CNN.

In [ ]:
def unsharp(img, alpha=1.0, sigma=1.0):
    blurred = ndi.gaussian_filter(img, sigma=sigma)
    return np.clip(img + alpha * (img - blurred), 0, 1)


def zscore_1ch(img):
    return ((img - img.mean()) / (img.std() + 1e-6))[None]


def clip199_1ch(img):
    return clip_minmax(img, lo=1, hi=99)[None]


def unsharp_sobel_199(img):
    img = clip_minmax(img, lo=1, hi=99)
    return np.stack([img, unsharp(img), norm(sobel_mag(img))], axis=0)


def rank_sobel_199(img):
    img = clip_minmax(img, lo=1, hi=99)
    return np.stack([img, rank_filter(img), norm(sobel_mag(img))], axis=0)


def rank_sobel_298(img):
    img = clip_minmax(img, lo=2, hi=98)
    return np.stack([img, rank_filter(img), norm(sobel_mag(img))], axis=0)


def robust_rank_sobel(img):
    img = scale_robust(img)
    return np.stack([img, rank_filter(img), norm(sobel_mag(img))], axis=0)


class Inputs(Dataset):
    def __init__(self, files, prepare):
        self.files, self.prepare = files, prepare

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        return torch.from_numpy(self.prepare(load_and_resize(self.files[i]))).float()

## Architectures

**Step 1, default CNN.** Four blocks of a 3 × 3 × 3 convolution, BatchNorm, ReLU, max pooling and dropout, with 64, 64, 128 and 256 filters, then global average pooling and a fully connected layer to 512 features.

In [ ]:
class CNN_Default(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, 3, padding=1); self.bn1 = nn.BatchNorm3d(64)
        self.conv2 = nn.Conv3d(64, 64, 3, padding=1); self.bn2 = nn.BatchNorm3d(64)
        self.conv3 = nn.Conv3d(64, 128, 3, padding=1); self.bn3 = nn.BatchNorm3d(128)
        self.conv4 = nn.Conv3d(128, 256, 3, padding=1); self.bn4 = nn.BatchNorm3d(256)
        self.pool = nn.MaxPool3d(2); self.drop = nn.Dropout3d(0.3)
        self.gap = nn.AdaptiveAvgPool3d(1); self.fc = nn.Linear(256, 512)

    def forward(self, x):
        x = self.drop(self.pool(F.relu(self.bn1(self.conv1(x)))))
        x = self.drop(self.pool(F.relu(self.bn2(self.conv2(x)))))
        x = self.drop(self.pool(F.relu(self.bn3(self.conv3(x)))))
        x = self.drop(self.pool(F.relu(self.bn4(self.conv4(x)))))
        x = self.gap(x).view(x.size(0), -1)
        return F.relu(self.fc(x))

**Step 2, multi scale readout.** The same encoder, read out at every block by adaptive average pooling to 2 × 2 × 2: 8 × (64 + 64 + 128 + 256) = 4,096 features.

In [ ]:
class CNN_MultiScale(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, 3, padding=1); self.bn1 = nn.BatchNorm3d(64)
        self.conv2 = nn.Conv3d(64, 64, 3, padding=1); self.bn2 = nn.BatchNorm3d(64)
        self.conv3 = nn.Conv3d(64, 128, 3, padding=1); self.bn3 = nn.BatchNorm3d(128)
        self.conv4 = nn.Conv3d(128, 256, 3, padding=1); self.bn4 = nn.BatchNorm3d(256)
        self.pool = nn.MaxPool3d(2); self.drop = nn.Dropout3d(0.3)
        self.avg_pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = self.drop(self.pool(F.relu(self.bn1(self.conv1(x)))))
        x2 = self.drop(self.pool(F.relu(self.bn2(self.conv2(x1)))))
        x3 = self.drop(self.pool(F.relu(self.bn3(self.conv3(x2)))))
        x4 = self.drop(self.pool(F.relu(self.bn4(self.conv4(x3)))))
        return torch.cat([self.avg_pool(xi).view(x.size(0), -1) for xi in [x1, x2, x3, x4]], dim=1)

**Step 3, GroupNorm and average pooling.** BatchNorm becomes GroupNorm with 8 groups, max pooling becomes average pooling, and dropout is removed.

In [ ]:
class CNN_GN_AvgPool(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, 3, padding=1); self.norm1 = nn.GroupNorm(8, 64)
        self.conv2 = nn.Conv3d(64, 64, 3, padding=1); self.norm2 = nn.GroupNorm(8, 64)
        self.conv3 = nn.Conv3d(64, 128, 3, padding=1); self.norm3 = nn.GroupNorm(8, 128)
        self.conv4 = nn.Conv3d(128, 256, 3, padding=1); self.norm4 = nn.GroupNorm(8, 256)
        self.pool = nn.AvgPool3d(2); self.avg_pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = self.pool(F.relu(self.norm1(self.conv1(x))))
        x2 = self.pool(F.relu(self.norm2(self.conv2(x1))))
        x3 = self.pool(F.relu(self.norm3(self.conv3(x2))))
        x4 = self.pool(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([self.avg_pool(xi).view(x.size(0), -1) for xi in [x1, x2, x3, x4]], dim=1)

**Step 5, depthwise separable convolutions** in blocks 2 to 4: a 3 × 3 × 3 filter per input channel followed by a 1 × 1 × 1 pointwise convolution.

In [ ]:
class CNN_DWSep(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, 3, padding=1); self.norm1 = nn.GroupNorm(8, 64)
        self.conv2 = DepthwiseSepConv3d(64, 64); self.norm2 = nn.GroupNorm(8, 64)
        self.conv3 = DepthwiseSepConv3d(64, 128); self.norm3 = nn.GroupNorm(8, 128)
        self.conv4 = DepthwiseSepConv3d(128, 256); self.norm4 = nn.GroupNorm(8, 256)
        self.pool = nn.AvgPool3d(2); self.avg_pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = self.pool(F.relu(self.norm1(self.conv1(x))))
        x2 = self.pool(F.relu(self.norm2(self.conv2(x1))))
        x3 = self.pool(F.relu(self.norm3(self.conv3(x2))))
        x4 = self.pool(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([self.avg_pool(xi).view(x.size(0), -1) for xi in [x1, x2, x3, x4]], dim=1)

**Step 7, double convolutions.** Two convolutions per block, each followed by GroupNorm and ReLU, with 64, 128, 256 and 512 channels: 8 × 960 = 7,680 features.

In [ ]:
class CNN_DoubleConv(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1a = nn.Conv3d(in_channels, 64, 3, padding=1); self.norm1a = nn.GroupNorm(8, 64)
        self.conv1b = nn.Conv3d(64, 64, 3, padding=1); self.norm1b = nn.GroupNorm(8, 64)
        self.conv2a = DepthwiseSepConv3d(64, 128); self.norm2a = nn.GroupNorm(8, 128)
        self.conv2b = DepthwiseSepConv3d(128, 128); self.norm2b = nn.GroupNorm(8, 128)
        self.conv3a = DepthwiseSepConv3d(128, 256); self.norm3a = nn.GroupNorm(8, 256)
        self.conv3b = DepthwiseSepConv3d(256, 256); self.norm3b = nn.GroupNorm(8, 256)
        self.conv4a = DepthwiseSepConv3d(256, 512); self.norm4a = nn.GroupNorm(8, 512)
        self.conv4b = DepthwiseSepConv3d(512, 512); self.norm4b = nn.GroupNorm(8, 512)
        self.pool = nn.AvgPool3d(2); self.avg_pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = F.relu(self.norm1a(self.conv1a(x))); x1 = self.pool(F.relu(self.norm1b(self.conv1b(x1))))
        x2 = F.relu(self.norm2a(self.conv2a(x1))); x2 = self.pool(F.relu(self.norm2b(self.conv2b(x2))))
        x3 = F.relu(self.norm3a(self.conv3a(x2))); x3 = self.pool(F.relu(self.norm3b(self.conv3b(x3))))
        x4 = F.relu(self.norm4a(self.conv4a(x3))); x4 = self.pool(F.relu(self.norm4b(self.conv4b(x4))))
        return torch.cat([self.avg_pool(xi).view(x.size(0), -1) for xi in [x1, x2, x3, x4]], dim=1)

**Step 10, covariance pooling.** Each block descriptor adds the 496 upper triangular entries of the covariance of its first 32 channels: 7,680 + 4 × 496 = 9,664 features. This is the un-CNN architecture, `uncnn.UnCNN`.

## Random forest evaluation

Five 90/10 train and test splits with seeds 0, 1, 2, 3 and 42, stratified by sex for classification. Random forests with 200 trees, maximum depth 6 and minimum split size 5; the classifier uses $\sqrt{d}$ features per split and balanced class weights, the regressor all $d$ features. Sex is scored by balanced accuracy and age by mean absolute error, reported as mean ± standard deviation across the five seeds.

In [ ]:
SEEDS = [0, 1, 2, 3, 42]


def eval_clf_seeds(X, y):
    scores = []
    for seed in SEEDS:
        cv, te = train_test_split(np.arange(len(y)), test_size=int(0.1 * len(y)), stratify=y, random_state=seed)
        rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_split=5, random_state=seed,
                                    n_jobs=1, max_features="sqrt", class_weight="balanced")
        rf.fit(X[cv], y[cv])
        scores.append(balanced_accuracy_score(y[te], rf.predict(X[te])))
    return scores


def eval_reg_seeds(X, y):
    maes = []
    for seed in SEEDS:
        cv, te = train_test_split(np.arange(len(y)), test_size=int(0.1 * len(y)), random_state=seed)
        rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_split=5, random_state=seed, n_jobs=1)
        rf.fit(X[cv], y[cv])
        maes.append(mean_absolute_error(y[te], rf.predict(X[te])))
    return maes


def extract(model_cls, prepare, in_ch):
    loader = DataLoader(Inputs(files, prepare), batch_size=1, num_workers=0)
    torch.manual_seed(0)
    model = model_cls(in_channels=in_ch).eval()
    with torch.no_grad():
        feats = np.vstack([model(x).numpy() for x in loader])
    del model
    gc.collect()
    return feats

## Ablation

In [ ]:
STEPS = [
    (1, "01_Default_CNN", CNN_Default, zscore_1ch, 1),
    (2, "02_+MultiScale", CNN_MultiScale, zscore_1ch, 1),
    (3, "03_+GroupNorm_AvgPool", CNN_GN_AvgPool, zscore_1ch, 1),
    (4, "04_+Clip199_MinMax", CNN_GN_AvgPool, clip199_1ch, 1),
    (5, "05_+DWSep", CNN_DWSep, clip199_1ch, 1),
    (6, "06_+3ch_Unsharp_Sobel", CNN_DWSep, unsharp_sobel_199, 3),
    (7, "07_+DoubleConv", CNN_DoubleConv, unsharp_sobel_199, 3),
    (8, "08_+RankFilter", CNN_DoubleConv, rank_sobel_199, 3),
    (9, "09_+Clip298", CNN_DoubleConv, rank_sobel_298, 3),
    (10, "10_+CovPool", UnCNN, rank_sobel_298, 3),
    (11, "11_+RobustScaler", UnCNN, robust_rank_sobel, 3),
]

rows, steps = [], []
for step, name, model_cls, prepare, in_ch in STEPS:
    t0 = time.time()
    X = extract(model_cls, prepare, in_ch)
    sex_scores, age_scores = eval_clf_seeds(X, sex), eval_reg_seeds(X, age)
    elapsed = time.time() - t0
    rows += [{"dataset": "NKI", "step": step, "model": name, "task": "sex", "metric": "balanced_accuracy",
              "seed": s, "value": v} for s, v in zip(SEEDS, sex_scores)]
    rows += [{"dataset": "NKI", "step": step, "model": name, "task": "age", "metric": "mae",
              "seed": s, "value": v} for s, v in zip(SEEDS, age_scores)]
    steps.append({"step": step, "model": name, "dim": X.shape[1], "runtime_s_per_img": elapsed / len(subjects),
                  "sex_mean": np.mean(sex_scores), "sex_sd": np.std(sex_scores),
                  "age_mean": np.mean(age_scores), "age_sd": np.std(age_scores)})
    print(f"{step:>2}  {name:<24} {X.shape[1]:>6}  sex {np.mean(sex_scores):.3f}  age {np.mean(age_scores):.2f}  "
          f"{elapsed / 60:.1f} min")
    del X
    gc.collect()

pd.DataFrame(rows).to_csv(OUT / "ablation_results.csv", index=False)
steps = pd.DataFrame(steps)
steps.to_csv(OUT / "steps.csv", index=False)

## Table II

In [ ]:
pd.DataFrame({
    "Model": steps["model"],
    "Final Dimension": steps["dim"].map("{:,}".format),
    "Ablation runtime (s/img)": steps["runtime_s_per_img"].round(1),
    "Sex Accuracy": (100 * steps["sex_mean"]).map("{:.1f}%".format),
    "Age MAE": steps["age_mean"].round(2),
})